In [1]:
import re
import requests
import geocoder
import time
import webbrowser
import sys


In [2]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

    

In [3]:
destination = input('wat is de supermarkt waar je heen wilt( vul supermarkt in of niks)')

In [22]:
destination = destination.lower()

if "alber" in destination or destination == "ah":
    supermarkt = "https://www.ah.nl/zoeken?query="

elif "plus" in destination:
    supermarkt = "https://www.plus.nl/zoekresultaten?SearchTerm="

elif "jum" in destination:
    supermarkt = "https://www.jumbo.com/producten/?searchType=keyword&searchTerms="

elif "spar" in destination:
    supermarkt = "https://www.spar.nl/zoek/?fq="

else:
    supermarkt = "https://google.com/search?q=supermarkt+voor+"

In [23]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

In [24]:
print(lat)
print(lon)

52.0908
5.1222


In [25]:
prompts = [
    f"""
Genereer EXACT 5 verschillende gerechten op basis van deze ingrediënten:

{items}

BELANGRIJKE REGELS:
- Geef EXACT 5 regels terug.
- Geen uitleg voor of na de regels.
- Geen markdown.
- Geen opsommingstekens.
- Elke regel moet EXACT dit formaat hebben:

gerecht_nr | gerecht | ingredienten | duur | nieuw ingredient = (ingredient)

VOORBEELD:
1 | Pasta met kip | kip, pasta, ui | 25 min | nieuw ingredient = (parmezaanse kaas)

EXTRA REGELS:
- Gebruik zoveel mogelijk ingrediënten uit de opgegeven lijst.
- Maximaal 1 nieuw ingrediënt per gerecht.
- Bereidingstijd maximaal 30 minuten.
- Het nieuwe ingrediënt MOET tussen haakjes staan.
- De tekst 'nieuw ingredient = ...' moet altijd het laatste onderdeel van de regel zijn.
- Gebruik het pipe-teken '|' uitsluitend als scheidingsteken.
- Gebruik geen pipe-tekens in gerechtnamen of ingrediënten.
- Nummer de gerechten van 1 t/m 5.

Geef alleen de 5 regels terug.
"""
]

In [26]:
for prompt in prompts:
    start_time = time.time()

In [27]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


In [35]:
url = 'http://127.0.0.1:1234/v1/chat/completions'

def lmbot():
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()

        end_time = time.time()

        result = response.json()

        answer = result["choices"][0]["message"]["content"]
        model_name = result.get("model", "onbekend")
        response_length = len(answer)

    except requests.exceptions.RequestException as e:
        print(f"Fout bij API-aanroep: {e}")


    for answe in answer.splitlines():
        if "nieuw ingredient" in answe.lower() or "nieuw ingrediënt" in answe.lower():
            ingredient = answe.split("=", 1)[1].strip()

            ingredient = ingredient.strip("()[]{}")
            ingredient = ingredient.replace(' ', '+')

            url2 = supermarkt+ingredient
            answe += f"|{url2}"

        print(answe)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

In [36]:
while True:
    lmbot()

    check = input("Zijn dit ingrediënten die je wilt? (Ja/Nee) ")

    if check.lower() == "ja":
        break


1 | Kip met ui en spec | kip, ei, spec, kaas | 20 min | nieuw ingredient = (parmezaanse kaas)|https://www.ah.nl/zoeken?query=parmezaanse+kaas
2 | Kip hotdog soep | kip, 1/3 hotdog, sla, ui | 25 min | nieuw ingredient = (soep)|https://www.ah.nl/zoeken?query=soep
3 | Ei spec kaas | ei, spec, kaas | 20 min | nieuw ingredient = (soep)|https://www.ah.nl/zoeken?query=soep
4 | Kip ui sla hotdog | kip, ui, sla, 1/3 hotdog | 25 min | nieuw ingredient = (ketchup)|https://www.ah.nl/zoeken?query=ketchup
5 | Ei spec kaas soep | ei, spec, kaas | 15 min | nieuw ingredient = (soep)|https://www.ah.nl/zoeken?query=soep

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 414 karakters
Tijd: 1085.95 seconden
--------------------------------------------------


In [37]:
if destination in ("niks", ""):
    sys.exit()

else:
    url2 = (
        f"https://www.google.com/maps/dir/?api=1"
        f"&origin={lat},{lon}"
        f"&destination={destination}"
        f"&travelmode=driving"
    )

    webbrowser.open(url2)